# PRODUCTION

In [20]:
### ==== IMPORT ====
import pypdf
import logging
from pathlib import Path

# Suppress all pypdf log messages below the ERROR level
logging.getLogger("pypdf").setLevel(logging.ERROR)

In [21]:
### ==== FUNCTIONS ====
# Count the number of TOTAL pages in the PDF file
def count_tot_pages(reader) -> int:
    """Counts the number of pages in the PDF file.

    RETURNS: 
    The total number of pages in the PDF file"""
    num_tot_pages = len(reader.pages)
    
    return num_tot_pages

# Count the number of words in the PDF file
def word_count(reader, last_page) -> int:
    """Count the number of words in the PDF file."""
    tot_words = 0

    for page in reader.pages[:last_page]:
        text = page.extract_text()
        tot_words += len(text.split())
    return tot_words

def count_cont_pages_OLD(reader) -> int:
    """Counts the number of pages in the PDF file that contain content."""
    num_cont_pages = 0

    for page in reader.pages:
        text = page.extract_text()
        if text and text.strip():  # Check if text is not empty or just whitespace
            num_cont_pages += 1

    return num_cont_pages


def print_pdf_text(reader) -> None:
    """Prints the text content of each page in the PDF file."""
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        print(f"Page {i + 1}:\n{text}\n{'-' * 40}")

In [22]:
def count_cont_pages(reader, audit=False, page_prints=False) -> int:
    """Counts the number of pages in the PDF file that contain main content."""
    num_cont_pages = 0
    num_tot_pages = len(reader.pages)
    min_end_page = max(1, int(num_tot_pages * 0.30))
    
    end_boundary_exact = {
        "references",
        "bibliography",
        "works cited",
        "list of references",
        "reference list",
        "appendix",
        "appendices",
        "referencer",
        "bibliografi",
        "litteratur",
        "litteraturliste",
        "litteraturfortegnelse",
        "kildeliste",
        "bilag",
        "appendiks",
        "list of figures",
        "list of tables",
    }
    end_boundary_prefix = (
        "references",
        "bibliography",
        "works cited",
        "appendix",
        "appendices",
        "referencer",
        "bibliografi",
        "litteratur",
        "kildeliste",
        "bilag",
        "appendiks",
    )

    boundary_page_text = None
    boundary_page_number = None

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        lines = [line.strip().lower() for line in text.splitlines() if line.strip()]

        matched_line = None
        match_trigger = None
        reject_reason = None

        for line in lines:
            tokens = line.split()
            prefix_token = None
            core_line = line

            # Allow heading labels like "6 References" or "F List of tables".
            if tokens:
                first_token = tokens[0].rstrip(").:-")
                if first_token.isdigit() or (len(first_token) == 1 and first_token.isalpha()):
                    prefix_token = first_token
                    core_line = " ".join(tokens[1:]).strip()

            # Determine which trigger matched (and record it).
            if core_line and core_line in end_boundary_exact:
                if prefix_token and prefix_token.isdigit():
                    match_trigger = f"numeric-prefix exact  ('{prefix_token} {core_line}')"
                elif prefix_token:
                    match_trigger = f"letter-prefix exact  ('{prefix_token} {core_line}')"
                else:
                    match_trigger = f"exact  ('{core_line}')"
            elif core_line and any(core_line.startswith(p) for p in end_boundary_prefix):
                matched_prefix = next(p for p in end_boundary_prefix if core_line.startswith(p))
                if prefix_token and prefix_token.isdigit():
                    match_trigger = f"numeric-prefix prefix-match  ('{prefix_token} {core_line}', prefix='{matched_prefix}')"
                elif prefix_token:
                    match_trigger = f"letter-prefix prefix-match  ('{prefix_token} {core_line}', prefix='{matched_prefix}')"
                else:
                    match_trigger = f"prefix-match  ('{core_line}', prefix='{matched_prefix}')"
            else:
                match_trigger = None

            if match_trigger is None:
                continue

            words = line.split()
            has_short_length = len(line) <= 60 and len(words) <= 8

            # Reject lines that look like running prose rather than headings.
            ends_with_comma_semicolon = line.endswith(",") or line.endswith(";") or line.endswith(":") or line.endswith(".") or line.endswith(")")

            core_words = core_line.split()
            first_core_token = core_words[0] if core_words else ""
            if first_core_token in end_boundary_exact:
                trailing_words = core_words[1:]
            else:
                trailing_words = core_words
            lowercase_trailing_count = sum(
                1 for w in trailing_words if w.isalpha() and w.islower()
            )
            has_many_lowercase_trailing = lowercase_trailing_count >= 4

            if not has_short_length:
                reject_reason = "failed length rule (>60 chars or >8 words)"
                if audit:
                    print(f"[AUDIT] Rejected candidate on page {page_number}: '{line}' ({reject_reason})")
                continue

            if ends_with_comma_semicolon:
                reject_reason = "ends with comma/semicolon/colon/period/parenthesis (sentence-like)"
                if audit:
                    print(f"[AUDIT] Rejected candidate on page {page_number}: '{line}' ({reject_reason})")
                continue

            if has_many_lowercase_trailing:
                reject_reason = "many lowercase trailing words (sentence-like)"
                if audit:
                    print(f"[AUDIT] Rejected candidate on page {page_number}: '{line}' ({reject_reason})")
                continue

            matched_line = line
            break

        if matched_line is not None:
            if audit:
                print(f"[AUDIT] Candidate end boundary on page {page_number} via {match_trigger}")

            # Validate that end boundary is found after first 30% of pages.
            if page_number > min_end_page:
                boundary_page_text = text
                boundary_page_number = page_number
                if audit:
                    print(f"[AUDIT] Accepted boundary on page {page_number} (>{min_end_page} pages threshold).")
                break
            elif audit:
                print(f"[AUDIT] Ignored candidate on page {page_number} (must be > {min_end_page}).")

        num_cont_pages += 1

    # __________________
    # DELETE THIS LATER:
    if page_prints:
        choice = input("Reached the end of the main content. Press 1 to print the next page, 2 to stop: ").strip()

        if choice == "1" and boundary_page_text is not None:
            # Print the first detected non-main-content page.
            print(f"========== START OF PAGE {boundary_page_number} ==========")
            print(f"{boundary_page_text}\n{'-' * 40}")
            print("========== END OF PAGE ==========")
        elif choice == "1":
            print("No validated end boundary page was found.")
    # __________________

    return num_cont_pages, match_trigger


In [23]:
### ==== MAIN ====
# Opem and read in a PDF

def main(pdf_path):
    with open(pdf_path, 'rb') as file:
        try:
            reader = pypdf.PdfReader(file)
        except Exception as e:
            print(f"Error reading PDF file: {e}")
        
        # Number of pages in the PDF file
        num_tot_pages = count_tot_pages(reader)
        print(f"Total number of pages in PDF: {num_tot_pages}")

        # Number of pages with main content in the PDF file
        num_cont_pages, match_trigger, = count_cont_pages(reader, audit=False, page_prints=False) # audit=True/False to get audit prints, page_prints=True/False to get option to print end page/first excluded page
        print(f"Number of pages with main content in PDF: {num_cont_pages}")
        print(f"Accepted candidate reason: {match_trigger}")

        # Number of words in the PDF file
        num_words_full = word_count(reader, num_tot_pages)
        print(f"Total number of words in the full PDF: {num_words_full}")

        # Number of words in the main content of the PDF file
        num_words_cont = word_count(reader, num_cont_pages)
        print(f"Number of words in the main content of the PDF: {num_words_cont}")


In [24]:
### ==== FILE PATHS ====
folder_path = "../Data/Raw_test/"

pdf_files = sorted(Path(folder_path).glob("*.pdf"))

if not pdf_files:
    print(f"No PDF files found in: {folder_path}")
else:
    total_files = len(pdf_files)
    print(f"Found {total_files} PDF file(s) in: {folder_path}")

    user_input = input(f"How many files do you want to process? (1-{total_files}): ").strip()

    try:
        num_to_process = int(user_input)
        if num_to_process <= 0:
            raise ValueError
    except ValueError:
        raise SystemExit("Error: Invalid input. Please enter a positive integer. Execution terminated.")

    if num_to_process > total_files:
        choice = input(
            f"Error: Requested {num_to_process} files, but only {total_files} available.\n"
            "Type 'all' to process all files, or 'q' to terminate: "
        ).strip().lower()

        if choice == "all":
            num_to_process = total_files
        elif choice in {"q", "quit"}:
            raise SystemExit("Execution terminated by user. No files were processed.")
        else:
            raise SystemExit("Error: Invalid choice. Execution terminated without processing files.")

    for pdf_path in pdf_files[:num_to_process]:
        print(f"\n=== Processing: {pdf_path.name} ===")
        main(pdf_path)

Found 23 PDF file(s) in: ../Data/Raw_test/

=== Processing: 5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf ===
Total number of pages in PDF: 61
Number of pages with main content in PDF: 51
Accepted candidate reason: exact  ('references')
Total number of words in the full PDF: 20540
Number of words in the main content of the PDF: 17924


# DEVELOPMENT

## TO DO: 

- [x] get pdf page count (incl/excl. appendix) \check
- [x] get word count \check
- [] rewrite for GCP and append results to csv file
- [] get supervisor(s)

In [ ]:
test_file = "../Data/RAW_test/5d1c8d79d9001d146569a4dc_Deep speech recognition in Danish (translated Dyb talegenkendelse pa dansk).pdf"
test_file = "../Data/RAW_test/5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf"
with open(test_file, 'rb') as file:
    try:
        reader = pypdf.PdfReader(file)
    except Exception as e:
        print(f"Error reading PDF file: {e}")

    # display page number
    number = 51
    page = reader.pages[number]
    text = page.extract_text()
    print(f"Page {number}:\n{text}\n{'-' * 40}")

In [ ]:
# Functions



# Count the number of CONTENT pages in the PDF file (excluding all formality pages)
def count_content_pages(reader) -> int:
    """Count all content pages after title page, abstract, and table of contents and ends before references and appendices.

    RETURNS:
    the total number of contentpages excluding all formality pages.
    """
    #num_content_pages = #TODO

    return #num_content_pages


# Count the number of words in the PDF file
def word_count(reader) -> int:
    """Count the number of words in the PDF file."""
    total_words_full_pdf = 0
    total_words_content_pdf = 0
    for i in range(2):
        for page in reader.pages:
            text = page.extract_text()
            if i == 1:
                total_words_full_pdf += len(text.split())
            else:
                total_words_content_pdf += len(text.split())
    return total_words_full_pdf, total_words_content_pdf


    

# CLI function
def main():
    input_path = Path("Data/RAW_test")
    file_name = "TEST.pdf" # to be made dynamic for user choise

    # Inital option for user:
    # load file
    pdf_file = input_path / file_name
    reader = open_pdf(pdf_file)

    # Options for user:
    # 1) Count pages in a single PDF file
    num_tot_pages = count_tot_pages(reader)
    print(f"Total number of pages in the PDF file: {num_tot_pages}")
    num_content_pages = count_content_pages(reader)
    print(f"Total number of content pages in the PDF file: {num_content_pages}")

    # 2) Count words in a single PDF file
    #word_count(reader)
    


In [ ]:
pdf_file = "Data/Raw_test/TEST.pdf"
# Options for user:
# 1) Count pages in a single PDF file
num_tot_pages = count_tot_pages(pdf_file)

In [ ]:




# Options for user:
# 1) Count pages in a single PDF file
num_tot_pages = count_tot_pages(reader)
print(f"Total number of pages in the PDF file: {num_tot_pages}")
num_content_pages = count_content_pages(reader)
print(f"Total number of content pages in the PDF file: {num_content_pages}")